In [0]:
import sys
sys.path.insert(0, "../lib")

import utils
import ingestors

catalog       = "workspace"
schemaname    = "silver"
volume_schema = "upsell"
tablename     = dbutils.widgets.get("tablename")
idfield       = dbutils.widgets.get("id_field")

In [0]:
remove_checkpoint = False
# Verifica se a tabela existe na camada silver.
if not utils.table_exists(spark, "workspace", "silver", tablename):
    print("Criando a tabela", tablename)
    query = utils.import_query(f"{tablename}.sql")
    (spark.sql(query)
          .write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .option("delta.enableChangeDataFeed", "true")
          .saveAsTable(f"silver.{tablename}"))
    
    remove_checkpoint = True

In [0]:
print("Iniciando CDF...")
ingest = ingestors.IngestorCDF(spark=spark,
                               catalog=catalog,
                               schemaname=schemaname,
                               tablename=tablename,
                               id_field=idfield)

if remove_checkpoint:
    dbutils.fs.rm(ingest.checkpoint, True)

stream = ingest.execute()
print("Ok.")

In [0]:
%sql

SELECT COUNT(*) FROM bronze.transactions

UNION ALL

SELECT COUNT(*)FROM silver.transacao

In [0]:
%sql

drop table silver.cliente